# Super Mario Bros — Ejecutar modelo pre-entrenado (uvipen/PPO-pytorch)
Notebook corregido para Python 3.8 + gym 0.26 + RTX 4060

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import torch
import torch.nn.functional as F
import numpy as np
import time

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

/home/deynar/miniconda3/envs/mario_uvipen/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 1.13.1+cu117
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# ── Verificar carpeta correcta ───────────────────────────────────────────────
files = os.listdir('.')
assert 'trained_models' in files, 'ERROR: Ejecuta desde la carpeta raíz del repo uvipen'
assert 'src' in files, 'ERROR: No se encuentra src/'

models = os.listdir('trained_models')
print(f'Modelos disponibles ({len(models)} total):')
for m in sorted(models)[:15]:
    print(f'  {m}')
if len(models) > 15:
    print(f'  ... y {len(models)-15} más')

Modelos disponibles (31 total):
  ppo_super_mario_bros_1_1
  ppo_super_mario_bros_1_2
  ppo_super_mario_bros_1_3
  ppo_super_mario_bros_1_4
  ppo_super_mario_bros_2_1
  ppo_super_mario_bros_2_2
  ppo_super_mario_bros_2_3
  ppo_super_mario_bros_2_4
  ppo_super_mario_bros_3_1
  ppo_super_mario_bros_3_2
  ppo_super_mario_bros_3_3
  ppo_super_mario_bros_3_4
  ppo_super_mario_bros_4_1
  ppo_super_mario_bros_4_2
  ppo_super_mario_bros_4_3
  ... y 16 más


In [3]:
# ── Configuración ─────────────────────────────────────────────────────────────
WORLD       = 6        # Mundo (1-8)
STAGE       = 2        # Nivel (1-4)
ACTION_TYPE = 'simple' # 'simple' | 'right' | 'complex'
EPISODES    = 3
FRAME_DELAY = 0.05     # ~50 FPS
DETERMINISTIC = True

MODEL_PATH = f'trained_models/ppo_super_mario_bros_{WORLD}_{STAGE}'

if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f'✅ Modelo encontrado: {MODEL_PATH} ({size_mb:.1f} MB)')
else:
    print(f'❌ Modelo NO encontrado: {MODEL_PATH}')
    print('Modelos disponibles:')
    for m in sorted(os.listdir('trained_models')):
        print(f'  trained_models/{m}')

✅ Modelo encontrado: trained_models/ppo_super_mario_bros_6_2 (2.5 MB)


In [4]:
# ── Importar acciones ─────────────────────────────────────────────────────────
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT, COMPLEX_MOVEMENT, RIGHT_ONLY

if ACTION_TYPE == 'right':
    actions = RIGHT_ONLY
elif ACTION_TYPE == 'simple':
    actions = SIMPLE_MOVEMENT
else:
    actions = COMPLEX_MOVEMENT

print(f'Acciones: {ACTION_TYPE} ({len(actions)} acciones)')
print(f'Nivel: W{WORLD}-{STAGE}')

Acciones: simple (7 acciones)
Nivel: W6-2


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [5]:
# ── Crear entorno manualmente (fix gym 0.26 TimeLimit + shape correcto) ───────
import gym
import gym_super_mario_bros
import gym.wrappers.time_limit as _tl
from nes_py.wrappers import JoypadSpace
from src.env import CustomReward, CustomSkipFrame

# Crear env base SIN TimeLimit (gym 0.26 lo agrega y rompe 4-tuple de nes-py)
try:
    env = gym_super_mario_bros.make(
        f'SuperMarioBros-{WORLD}-{STAGE}-v0',
        max_episode_steps=None
    )
except TypeError:
    env = gym_super_mario_bros.make(f'SuperMarioBros-{WORLD}-{STAGE}-v0')

# Quitar TimeLimit si gym lo agrego igual
while isinstance(env, _tl.TimeLimit):
    env = env.env

env = JoypadSpace(env, actions)
env = CustomReward(env, world=WORLD, stage=STAGE)
env = CustomSkipFrame(env, skip=4)  # <- 4 frames apilados

print(f'Observation space: {env.observation_space.shape}')  # debe ser (4, 84, 84)
print(f'Action space: {env.action_space.n}')

# Verificar shape
state_test = env.reset()
print(f'State shape tras reset: {state_test.shape}')  # debe ser (1, 4, 84, 84)
print(f'State dtype: {state_test.dtype}')             # debe ser float32

/home/deynar/miniconda3/envs/mario_uvipen/lib/python3.8/site-packages/gym/envs/registration.py:555: UserWarning: WARN: The environment SuperMarioBros-6-2-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(


Observation space: (4, 84, 84)
Action space: 7
State shape tras reset: (1, 4, 84, 84)
State dtype: float32


/home/deynar/miniconda3/envs/mario_uvipen/lib/python3.8/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(


In [6]:
# ── Cargar modelo ─────────────────────────────────────────────────────────────
from src.model import PPO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando device: {device}')

model = PPO(env.observation_space.shape[0], env.action_space.n)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

print(f'✅ Modelo cargado en {device}')
print(f'   Input channels: {env.observation_space.shape[0]}')
print(f'   Output actions: {env.action_space.n}')

Usando device: cuda
✅ Modelo cargado en cuda
   Input channels: 4
   Output actions: 7


In [7]:
# ── Helper: preparar estado ───────────────────────────────────────────────────
def prepare_state(state_np):
    """
    Convierte numpy array del env al tensor que espera el modelo.
    El modelo espera: (batch=1, canales=4, H=84, W=84) float32 en device.
    """
    t = torch.from_numpy(state_np).float()
    # Asegurar 4 dimensiones (batch, C, H, W)
    if t.dim() == 3:   # (C, H, W) -> agregar batch
        t = t.unsqueeze(0)
    elif t.dim() == 4 and t.shape[0] != 1:
        t = t[:1]      # tomar solo el primer batch si hay varios
    return t.to(device)

# Test rapido
state_test = env.reset()
tensor_test = prepare_state(state_test)
print(f'Tensor shape para modelo: {tensor_test.shape}')  # debe ser [1, 4, 84, 84]
with torch.no_grad():
    logits, value = model(tensor_test)
print(f'✅ Forward pass OK — logits shape: {logits.shape}')

Tensor shape para modelo: torch.Size([1, 4, 84, 84])
✅ Forward pass OK — logits shape: torch.Size([1, 7])


In [8]:
# ── Jugar con ventana visual ──────────────────────────────────────────────────
scores = []

for ep in range(1, EPISODES + 1):
    state     = prepare_state(env.reset())
    done      = False
    score     = 0.0
    step      = 0
    completed = False

    print(f'\n═══ Episodio {ep}/{EPISODES} ═══')

    while not done:
        # Render — abre/actualiza ventana del emulador
        env.render()

        # Inferencia
        with torch.no_grad():
            logits, value = model(state)
            policy = F.softmax(logits, dim=1)

        if DETERMINISTIC:
            action = torch.argmax(policy).item()
        else:
            action = torch.multinomial(policy, 1).item()

        # Step — fix 4/5 tuple ya aplicado en env.py
        state_np, reward, done, info = env.step(action)
        state  = prepare_state(state_np)
        score += reward
        step  += 1

        if step % 100 == 0:
            x_pos = info.get('x_pos', 0)
            print(f'  Step {step:4d} | Score: {score:7.1f} | X: {x_pos} | '
                  f'Vida: {info.get("life", "?")}', end='\r')

        if info.get('flag_get', False):
            completed = True
            print(f'\n  🏁 ¡NIVEL COMPLETADO! Score: {score:.0f}')
            time.sleep(1.5)
            break

        time.sleep(FRAME_DELAY)

    scores.append(score)
    status = '✅ COMPLETADO' if completed else '❌ Murió'
    print(f'\n  {status} | Score final: {score:.0f} | Steps: {step}')

print(f"\n{'='*40}")
print(f'Score medio : {np.mean(scores):.1f}')
print(f'Score máximo: {np.max(scores):.1f}')
print(f'Score mínimo: {np.min(scores):.1f}')


═══ Episodio 1/3 ═══


/home/deynar/miniconda3/envs/mario_uvipen/lib/python3.8/site-packages/gym/utils/passive_env_checker.py:272: UserWarning: WARN: No render modes was declared in the environment (env.metadata['render_modes'] is None or not defined), you may have trouble when calling `.render()`.
  logger.warn(
/home/deynar/miniconda3/envs/mario_uvipen/lib/python3.8/site-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(
/home/deynar/miniconda3/envs/mario_uvipen/lib/python3.8/site-packages/gym/utils/passive_env_checker.py:225: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(done, (bool, np.bool8)):


  Step  300 | Score:   268.7 | X: 2785 | Vida: 2
  🏁 ¡NIVEL COMPLETADO! Score: 340

  ✅ COMPLETADO | Score final: 340 | Steps: 366

═══ Episodio 2/3 ═══


KeyboardInterrupt: 

In [9]:
env.close()
print('Entorno cerrado.')

Entorno cerrado.
